# Лабораторная работа 6а. Задачи по теме «Самовнимание»

Обозначения (как в стандартном описании блока самовнимания): входная матрица $X\in\mathbb R^{D\times N}$ (столбцы — $N$ векторов длины $D$);
$$Q=\beta_q\mathbf 1^T+\Omega_qX,\quad K=\beta_k\mathbf 1^T+\Omega_kX,\quad V=\beta_v\mathbf 1^T+\Omega_vX,$$
$$\mathrm{Sa}[X]=V\cdot\mathrm{Softmax}\!\left[\tfrac{K^TQ}{\sqrt D}\right],$$
где $\mathrm{Softmax}$ применяется к каждому столбцу матрицы $N\times N$ отдельно. Масштаб $1/\sqrt D$ на результаты не влияет.

## Задача 1. Число параметров

**Значения, ключи, запросы.** Каждое из преобразований $V,K,Q$ — аффинное отображение $\mathbb R^D\to\mathbb R^D$: матрица $D\times D$ весов и вектор смещений длины $D$, т. е. $D^2+D$ параметров. Три преобразования дают
$$\boxed{3(D^2+D)=3D^2+3D}$$
параметров. Важно: это число **не зависит от $N$** — одни и те же веса применяются к каждому из $N$ векторов (разделение параметров).

**Матрица внимания.** $A=\mathrm{Softmax}[K^TQ/\sqrt D]$ имеет размер $N\times N$, т. е. $N^2$ элементов, но **обучаемых параметров в ней нет**: она целиком вычисляется из входа $X$ (скалярные произведения и softmax не содержат весов). Число её элементов растёт квадратично с длиной последовательности — это влияет на память и время вычислений, а не на число параметров.

**Аналогичный перцептрон.** Полносвязный слой, отображающий $DN$ входов в $DN$ выходов, содержит матрицу $DN\times DN$ и $DN$ смещений: $(DN)^2+DN$ параметров. Если брать «двуслойный» перцептрон в буквальном смысле (два полносвязных слоя, скрытый слой ширины $DN$), получим
$$\boxed{2\big[(DN)^2+DN\big]=2D^2N^2+2DN}$$
(для одного слоя — вдвое меньше; порядок $D^2N^2$ в обоих случаях). Параметры растут квадратично по $N$, тогда как у самовнимания не зависят от $N$.

In [1]:
def params_attention(D): return 3*(D**2 + D)
def params_fc_one(D, N): return (D*N)**2 + D*N
def params_fc_two(D, N): return 2*((D*N)**2 + D*N)

import pandas as pd
rows = []
for D, N in [(4, 3), (64, 16), (512, 128), (1024, 1000)]:
    rows.append(dict(D=D, N=N, **{"самовнимание 3(D²+D)": params_attention(D), "элементов в матрице внимания N²": N**2,
                                   "FC, 1 слой": params_fc_one(D, N), "FC, 2 слоя": params_fc_two(D, N),
                                   "отношение FC(2)/Sa": round(params_fc_two(D, N)/params_attention(D), 1)}))
pd.DataFrame(rows).set_index(["D", "N"])

,,самовнимание 3(D²+D),элементов в матрице внимания N²,"FC, 1 слой","FC, 2 слоя",отношение FC(2)/Sa
D,N,,,,,
4,3,60,9,156,312,5.2
64,16,12480,256,1049600,2099200,168.2
512,128,787968,16384,4295032832,8590065664,10901.5
1024,1000,3148800,1000000,1048577024000,2097154048000,666016.9


In [2]:
# Проверка подсчёта на реальных тензорах: считаем размеры массивов параметров
import numpy as np
D, N = 6, 5
sa = {"Wq": (D, D), "bq": (D,), "Wk": (D, D), "bk": (D,), "Wv": (D, D), "bv": (D,)}
print("Sa: параметров =", sum(int(np.prod(s)) for s in sa.values()), "| формула:", params_attention(D))
fc = {"W1": (D*N, D*N), "b1": (D*N,), "W2": (D*N, D*N), "b2": (D*N,)}
print("FC (2 слоя): параметров =", sum(int(np.prod(s)) for s in fc.values()), "| формула:", params_fc_two(D, N))

Sa: параметров = 126 | формула: 126
FC (2 слоя): параметров = 1860 | формула: 1860


## Задача 2. Эквивариантность самовнимания относительно перестановок

Пусть $P$ — матрица перестановки: $P^TP=PP^T=I$, а в каждом столбце и каждой строке ровно одна единица; значит $\mathbf 1^TP=\mathbf 1^T$. Умножение справа на $P$ переставляет **столбцы** матрицы $X$ (т. е. порядок входных векторов).

**Шаг 1. Значения, ключи, запросы.** 
$$V[XP]=\beta_v\mathbf 1^T+\Omega_vXP=(\beta_v\mathbf 1^T)P+\Omega_vXP=V[X]P,$$
так как $\mathbf 1^TP=\mathbf 1^T$. Аналогично $K[XP]=KP$, $Q[XP]=QP$.

**Шаг 2. Матрица оценок.** $S'=(KP)^T(QP)=P^T(K^TQ)P=P^TSP$, где $S=K^TQ$.

**Шаг 3. Лемма: $\mathrm{Softmax}[P^TSP]=P^T\,\mathrm{Softmax}[S]\,P$.**
- Умножение $S$ справа на $P$ переставляет столбцы $S$; softmax действует на каждый столбец независимо, поэтому $\mathrm{Softmax}[SP]=\mathrm{Softmax}[S]P$.
- Умножение слева на $P^T$ переставляет элементы внутри каждого столбца. Для вектора $z$: $\mathrm{softmax}_i(Pz)=e^{(Pz)_i}/\sum_j e^{(Pz)_j}$, а сумма в знаменателе не зависит от порядка слагаемых, поэтому $\mathrm{softmax}(P^Tz)=P^T\mathrm{softmax}(z)$. Значит $\mathrm{Softmax}[P^T(SP)]=P^T\mathrm{Softmax}[SP]=P^T\mathrm{Softmax}[S]P$.

**Шаг 4. Выход.** 
$$\mathrm{Sa}[XP]=V[XP]\,\mathrm{Softmax}[S']=(VP)\,P^T\mathrm{Softmax}[S]\,P=V\underbrace{PP^T}_{I}\mathrm{Softmax}[S]\,P=\mathrm{Sa}[X]\,P.\qquad\blacksquare$$

Смысл: если переставить входные векторы, выходные векторы переставятся так же — блок самовнимания не знает о порядке токенов (поэтому в трансформерах добавляют позиционное кодирование). Проверим численно.

In [3]:
rng = np.random.default_rng(0)
def softmax_cols(S):
    E = np.exp(S - S.max(axis=0, keepdims=True)); return E/E.sum(axis=0, keepdims=True)
def Sa(X, p):
    D = X.shape[0]; ones = np.ones((1, X.shape[1]))
    Q = p["bq"][:, None]@ones + p["Wq"]@X; K = p["bk"][:, None]@ones + p["Wk"]@X; V = p["bv"][:, None]@ones + p["Wv"]@X
    return V @ softmax_cols(K.T@Q/np.sqrt(D))

D, N = 3, 4
p = {k: rng.normal(size=(D, D)) if k[0] == "W" else rng.normal(size=D) for k in ["Wq", "bq", "Wk", "bk", "Wv", "bv"]}
X = rng.normal(size=(D, N))
P = np.array([[0,0,1,0],[1,0,0,0],[0,0,0,1],[0,1,0,0]], float)       # матрица из условия
print("P^T P = I:", np.allclose(P.T@P, np.eye(4)))
print("max|Sa(XP) - Sa(X)P| (матрица из условия) =", np.abs(Sa(X@P, p) - Sa(X, p)@P).max())
errs = []
for _ in range(1000):
    Pr = np.eye(N)[:, rng.permutation(N)]; Xr = rng.normal(size=(D, N))
    errs.append(np.abs(Sa(Xr@Pr, p) - Sa(Xr, p)@Pr).max())
print("1000 случайных перестановок: макс. ошибка =", max(errs))
# контрпример для сравнения: полносвязный слой НЕ эквивариантен
W = rng.normal(size=(D*N, D*N)); fc = lambda Xm: (W@Xm.reshape(-1, order="F")).reshape(D, N, order="F")
print("Для полносвязного слоя: max|f(XP) - f(X)P| =", np.abs(fc(X@P) - fc(X)@P).max())

P^T P = I: True
max|Sa(XP) - Sa(X)P| (матрица из условия) = 2.220446049250313e-16
1000 случайных перестановок: макс. ошибка = 8.881784197001252e-16
Для полносвязного слоя: max|f(XP) - f(X)P| = 13.97583304006748


## Задача 3. Производные softmax

$y_i=\dfrac{e^{z_i}}{\sum_j e^{z_j}}$. При $j=i$: $\dfrac{\partial y_i}{\partial z_i}=y_i-y_i^2$; при $j\ne i$: $\dfrac{\partial y_i}{\partial z_j}=-y_iy_j$. Компактно:
$$\boxed{\frac{\partial y_i}{\partial z_j}=y_i(\delta_{ij}-y_j)}$$
Для $z=(-3,\,1,\,100,\,5,\,-1)$ вычисляем численно. Чтобы не потерять малые значения, диагональ считаем как $y_i\sum_{k\ne i}y_k$ (вместо $y_i(1-y_i)$, где $1-y_3$ при $y_3\approx1$ даёт катастрофическое сокращение). Результат проверяем арифметикой высокой точности (`decimal`, 60 знаков).

In [4]:
z = np.array([-3., 1., 100., 5., -1.])
e = np.exp(z - z.max()); y = e/e.sum()
other = y.sum() - y                 # неточно для y3; поправим ниже
other = np.array([np.delete(y, i).sum() for i in range(5)])   # Σ_{k≠i} y_k без сокращения
J = -np.outer(y, y); J[np.diag_indices(5)] = y*other
print("y =", y)
np.set_printoptions(linewidth=160, precision=3)
print("\nМатрица производных J[i,j] = ∂y_i/∂z_j:"); print(J)

y = [1.85211677e-45 1.01122149e-43 1.00000000e+00 5.52108228e-42
 1.36853947e-44]

Матрица производных J[i,j] = ∂y_i/∂z_j:
[[ 1.852e-45 -1.873e-88 -1.852e-45 -1.023e-86 -2.535e-89]
 [-1.873e-88  1.011e-43 -1.011e-43 -5.583e-85 -1.384e-87]
 [-1.852e-45 -1.011e-43  5.638e-42 -5.521e-42 -1.369e-44]
 [-1.023e-86 -5.583e-85 -5.521e-42  5.521e-42 -7.556e-86]
 [-2.535e-89 -1.384e-87 -1.369e-44 -7.556e-86  1.369e-44]]


In [5]:
from decimal import Decimal, getcontext
getcontext().prec = 60
zd = [Decimal(v) for v in (-3, 1, 100, 5, -1)]
ed = [v.exp() for v in zd]; sd = sum(ed); yd = [v/sd for v in ed]
Jd = [[yd[i]*((1 if i == j else 0) - yd[j]) for j in range(5)] for i in range(5)]
rel = max(abs(float(Jd[i][j]) - J[i, j])/abs(float(Jd[i][j])) for i in range(5) for j in range(5))
print("Макс. относительное расхождение с вычислением высокой точности:", rel)
import pandas as pd
pd.DataFrame(J, index=[f"y{i+1}" for i in range(5)], columns=[f"z{j+1}" for j in range(5)]).map(lambda v: f"{v:.3e}")

Макс. относительное расхождение с вычислением высокой точности: 1.8909058506768062e-16


,z1,z2,z3,z4,z5
y1,1.852e-45,-1.873e-88,-1.852e-45,-1.023e-86,-2.535e-89
y2,-1.873e-88,1.011e-43,-1.011e-43,-5.583e-85,-1.384e-87
y3,-1.852e-45,-1.011e-43,5.638e-42,-5.521e-42,-1.369e-44
y4,-1.023e-86,-5.583e-85,-5.521e-42,5.521e-42,-7.556e-86
y5,-2.535e-89,-1.384e-87,-1.369e-44,-7.556e-86,1.369e-44


In [6]:
print("Суммы по строкам (должны быть 0):", J.sum(axis=1))
print("Симметрия J = J^T:", np.allclose(J, J.T, rtol=1e-12, atol=0))
print("Знаки диагонали ≥ 0:", (np.diag(J) >= 0).all(), "| внедиагональные ≤ 0:", (J[~np.eye(5, dtype=bool)] <= 0).all())
print("Максимальная |производная| =", np.abs(J).max())

Суммы по строкам (должны быть 0): [-1.025e-86 -5.597e-85 -2.589e-58 -7.556e-86  0.000e+00]
Симметрия J = J^T: True
Знаки диагонали ≥ 0: True | внедиагональные ≤ 0: True
Максимальная |производная| = 5.6377419377708336e-42


### Выводы

1. **Все 25 производных практически равны нулю**: наибольшая по модулю $\approx5{,}6\cdot10^{-42}$ (элементы $J_{33},J_{34},J_{43},J_{44}$, связанные с $z_3$ и $z_4$); остальные ещё на несколько порядков меньше. Так как $z_3=100$ намного больше остальных, $y_3\approx1$, а $y_{1,2,4,5}\approx0$. Softmax насыщен: небольшое изменение любого $z_j$ практически не меняет выходы.
2. **Следствие для обучения:** градиент, проходящий через такой softmax, исчезает (затухающие градиенты) — параметры, формирующие $z$, почти не получают сигнала ошибки. Поэтому на входе softmax нужно следить за масштабом логитов (нормализация, масштаб $1/\sqrt D$ в самовнимании).
3. **Структура якобиана** (верна всегда): диагональ неотрицательна ($y_i(1-y_i)\ge0$), внедиагональные элементы неположительны ($-y_iy_j$) — рост одного логита увеличивает «свой» выход и уменьшает остальные; матрица симметрична; сумма по каждой строке и столбцу равна нулю, т. к. $\sum_iy_i=1$ — одновременный сдвиг всех $z_j$ на одну константу ничего не меняет (инвариантность softmax к сдвигу).

## Приложение. Использование генеративного ИИ (промт)

Задание выполнено с помощью Claude (Anthropic). Промт пользователя:

> Можешь еще эти лаб работы сделать пожалуйста
> (к промту были приложены файлы lab6a_1.pdf и lab6b_1.pdf; ранее в этом же диалоге: «Сделай эти две лаб работы. На выходе должно быть два notebook ... Если будете пользоваться, приложите к отчёту промты. ... Мне надо чтобы ты сделал все за меня ... Уложись в рамках лимита пожалуйста»)